# Problem 4 - Danh gia hieu suat nhan vien ban hang
Danh gia theo SalesPersonID, doanh thu, hoa don, xu huong, thai do phuc vu va chat luong giao dich.

In [ ]:
from pathlib import Path
import warnings
import unicodedata
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def locate_repo():
    for start in [Path.cwd().resolve(), Path(r'C:/dpbngoc/DA & AI/DAAI_N2.3')]:
        for path in [start, *start.parents]:
            if (path / 'silver_data').exists(): return path
    raise FileNotFoundError('Khong tim thay silver_data')

REPO_ROOT = locate_repo()
def find_data_file(filename):
    paths = sorted((REPO_ROOT / 'silver_data').glob(f'.silver_pipeline_work_*/excel/{filename}'), reverse=True)
    for path in paths + [REPO_ROOT / filename]:
        if path.exists(): return path
    raise FileNotFoundError(filename)

def show_result(df, name, index=False):
    print(f'\n{"="*100}\n{name.replace("_"," " ).replace(".csv","").upper()}\n{"="*100}')
    display(df)

POSITIVE_WORDS = ['chuyen nghiep','than thien','lich su','san sang ho tro','nhiet tinh','hai long','tot','nhanh','tuyet voi']
NEGATIVE_WORDS = ['te','cham','khong hai long','bat lich su','thai do kem','khong ho tro','that vong','phien','khieu nai']

def sentiment(text):
    value = ''.join(c for c in unicodedata.normalize('NFD', str(text).lower()) if unicodedata.category(c) != 'Mn')
    pos = sum(word in value for word in POSITIVE_WORDS)
    neg = sum(word in value for word in NEGATIVE_WORDS)
    return 'Positive' if pos > neg else ('Negative' if neg > pos else 'Neutral')

def longest_decline(values):
    best = current = 0
    for change in pd.Series(values).pct_change().iloc[1:]:
        current = current + 1 if pd.notna(change) and change < 0 else 0
        best = max(best, current)
    return best

def main():
    orders = pd.read_csv(find_data_file('orders_enriched.csv'), low_memory=False)
    items = pd.read_csv(find_data_file('order_items.csv'), low_memory=False)
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
    orders['order_status'] = orders['order_status'].astype(str).str.lower().str.strip()
    orders['sales_employee_id'] = orders['sales_employee_id'].astype(str).str.strip()
    orders['sales_employee_name'] = orders['sales_employee_name'].fillna('Unknown').astype(str)
    items[['quantity','unit_price','discount_amount']] = items[['quantity','unit_price','discount_amount']].apply(pd.to_numeric, errors='coerce')
    items['NetRevenue'] = (items['quantity'] * items['unit_price'] - items['discount_amount'].fillna(0)).clip(lower=0)
    order_revenue = items.groupby('order_id', as_index=False).agg(NetRevenue=('NetRevenue','sum'), Quantity=('quantity','sum'))
    data = orders.merge(order_revenue, on='order_id', how='left', validate='one_to_one')
    data[['NetRevenue','Quantity']] = data[['NetRevenue','Quantity']].fillna(0)
    data['IsSuccessful'] = data['order_status'].eq('delivered')
    data['IsReturned'] = data['order_status'].eq('returned')
    data['Sentiment'] = data['comment'].fillna('').map(sentiment)
    keys = ['sales_employee_id','sales_employee_name']

    employee = data.groupby(keys, as_index=False).agg(
        TotalInvoices=('order_id','nunique'), SuccessfulInvoices=('IsSuccessful','sum'),
        ReturnedInvoices=('IsReturned','sum'), CustomersHandled=('customer_id','nunique'))
    successful = data[data['IsSuccessful']].groupby(keys, as_index=False).agg(
        Revenue=('NetRevenue','sum'), SuccessfulQuantity=('Quantity','sum'))
    employee = employee.merge(successful, on=keys, how='left')
    employee[['Revenue','SuccessfulQuantity']] = employee[['Revenue','SuccessfulQuantity']].fillna(0)
    employee['CompletionRate_%'] = employee['SuccessfulInvoices'] / employee['TotalInvoices'].replace(0,np.nan) * 100
    employee['ReturnRate_%'] = employee['ReturnedInvoices'] / employee['TotalInvoices'].replace(0,np.nan) * 100
    employee['AvgSalesPerInvoice'] = employee['Revenue'] / employee['SuccessfulInvoices'].replace(0,np.nan)

    service_counts = data.pivot_table(index=keys, columns='Sentiment', values='order_id', aggfunc='nunique', fill_value=0).reset_index()
    for col in ['Positive','Neutral','Negative']:
        if col not in service_counts: service_counts[col] = 0
    service_counts['PositiveRate_%'] = service_counts['Positive'] / service_counts[['Positive','Neutral','Negative']].sum(axis=1).replace(0,np.nan) * 100
    service_counts['NegativeRate_%'] = service_counts['Negative'] / service_counts[['Positive','Neutral','Negative']].sum(axis=1).replace(0,np.nan) * 100
    employee = employee.merge(service_counts, on=keys, how='left')

    delivered = data[data['IsSuccessful']].copy()
    delivered['month'] = delivered['order_date'].dt.to_period('M').dt.to_timestamp()
    raw_trend = delivered.groupby(keys + ['month'], as_index=False).agg(Revenue=('NetRevenue','sum'), Invoices=('order_id','nunique'))
    months = pd.date_range(delivered['month'].min(), delivered['month'].max(), freq='MS')
    employees = employee[keys].drop_duplicates()
    grid = employees.assign(_k=1).merge(pd.DataFrame({'month':months,'_k':1}), on='_k').drop(columns='_k')
    trends = grid.merge(raw_trend, on=keys + ['month'], how='left').fillna({'Revenue':0,'Invoices':0})
    trend_rows = []
    for (emp_id, emp_name), group in trends.groupby(keys):
        group = group.sort_values('month'); values = group['Revenue'].to_numpy(float)
        mean = values.mean(); std = values.std(ddof=1); slope = np.polyfit(np.arange(len(values)), values, 1)[0]
        cv = std / mean if mean else np.nan; normalized_slope = slope / mean if mean else 0
        streak = longest_decline(values)
        status = 'Cai thien' if normalized_slope >= .01 else ('Suy giam' if normalized_slope <= -.01 or streak >= 2 else ('On dinh' if cv < .50 else 'Bien dong'))
        trend_rows.append({'sales_employee_id':emp_id,'sales_employee_name':emp_name,'MonthlyMeanRevenue':mean,'RevenueCV':cv,'MonthlySlope':slope,'NormalizedSlope':normalized_slope,'LongestDeclineStreak':streak,'TrendStatus':status})
    trend_metrics = pd.DataFrame(trend_rows)
    employee = employee.merge(trend_metrics, on=keys, how='left')

    employee['RevenueScore'] = employee['Revenue'].rank(pct=True)
    employee['CompletionScore'] = employee['CompletionRate_%'].rank(pct=True)
    employee['AOVScore'] = employee['AvgSalesPerInvoice'].rank(pct=True)
    employee['ServiceScore'] = employee['PositiveRate_%'].rank(pct=True)
    employee['StabilityScore'] = employee['RevenueCV'].rank(pct=True, ascending=False)
    employee['KPI_Score'] = 100 * (0.35*employee['RevenueScore'] + 0.25*employee['CompletionScore'] + 0.15*employee['AOVScore'] + 0.15*employee['ServiceScore'] + 0.10*employee['StabilityScore'])
    employee['PerformanceTier'] = pd.qcut(employee['KPI_Score'].rank(method='first'), q=4, labels=['Underperformer','Average','High','Top Performer'])
    employee['Recommendation'] = np.select([
        employee['PerformanceTier'].astype(str).eq('Top Performer'),
        employee['CompletionRate_%'] < employee['CompletionRate_%'].median(),
        employee['PositiveRate_%'] < employee['PositiveRate_%'].median(),
        employee['TrendStatus'].eq('Suy giam')],
        ['Thuong hieu suat; chia se best practice va giao khach hang gia tri cao',
         'Dao tao quan ly pipeline va ky nang chot don',
         'Coaching thai do phuc vu va xu ly phan hoi',
         'Lap ke hoach phuc hoi theo thang va coaching 1-1'],
        default='Duy tri KPI; dao tao nang cao AOV va cross-sell')
    employee['OverallRank'] = employee['KPI_Score'].rank(method='dense', ascending=False).astype(int)
    employee = employee.sort_values('OverallRank')

    show_result(employee, 'employee_kpi_ranking')
    show_result(employee.head(10), 'top_performers')
    show_result(employee.sort_values('KPI_Score').head(10), 'underperformers')
    show_result(trends, 'employee_monthly_trends_all')
    show_result(trend_metrics, 'employee_trend_stability')
    show_result(service_counts, 'employee_service_attitude')
    show_result(employee[['sales_employee_id','sales_employee_name','TotalInvoices','SuccessfulInvoices','Revenue','CompletionRate_%','AvgSalesPerInvoice','ReturnRate_%','PositiveRate_%','KPI_Score','OverallRank','Recommendation']], 'transaction_quality_and_actions')

    top_ids = employee.head(5)['sales_employee_id']
    plt.figure(figsize=(12,6))
    for emp_id, group in trends[trends['sales_employee_id'].isin(top_ids)].groupby('sales_employee_id'):
        plt.plot(group['month'], group['Revenue'], label=emp_id)
    plt.title('Xu huong doanh thu Top 5 nhan vien'); plt.ylabel('VND'); plt.legend(); plt.tight_layout()
    plt.show()
    plt.figure(figsize=(11,6)); view = employee.head(15).sort_values('KPI_Score')
    plt.barh(view['sales_employee_id'], view['KPI_Score']); plt.xlabel('KPI Score')
    plt.title('Top 15 nhan vien theo KPI tong hop'); plt.tight_layout()
    plt.show()
    plt.figure(figsize=(9,6)); plt.scatter(employee['SuccessfulInvoices'], employee['Revenue'], c=employee['KPI_Score'], cmap='viridis', alpha=.7)
    plt.xlabel('Hoa don thanh cong'); plt.ylabel('Doanh thu'); plt.title('So hoa don va doanh thu theo nhan vien'); plt.colorbar(label='KPI Score')
    plt.tight_layout(); plt.show()
    service_plot = service_counts.set_index('sales_employee_id')[['Positive','Neutral','Negative']].sum().to_frame('Comments')
    service_plot.plot(kind='bar', figsize=(8,5), legend=False, title='Tong hop thai do phuc vu qua sentiment')
    plt.tight_layout(); plt.show()
    print('Hoan tat Problem 4 | So nhan vien:', len(employee)); print('Tat ca bang va bieu do da hien thi truc tiep.')
    return employee, trends, service_counts

employee_result, employee_trends, service_result = main()
